# 02 · Colab A100 80GB 无头批处理量产（按独立项目隔离 + 断点续传 + 自动关机防烧卡）

- 只需指定 `PROJECT_ID`（如 `01_rainy_signal`），自动读取 `/MyDrive/AI_Video/projects/<PROJECT_ID>/tasks.csv`，并将所有成品视频与 `manifest.jsonl` 实时写回该项目 `outputs/` 目录。
- 默认开启 `AUTO_DISCONNECT = True`，整批任务落盘同步后自动释放 A100 计算单元。

In [ ]:
#@title 1. 挂载 5TB Google Drive、拉取代码并检查 GPU
REPO_URL = "https://github.com/WSQ233/AI-Video.git" #@param {type:"string"}
REPO_DIR = "/content/AI-Video" #@param {type:"string"}

!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)


In [ ]:
#@title 2. 选择独立项目并预热本次模型包到本地 SSD
PROJECT_ID = "01_rainy_signal" #@param {type:"string"}
ACTIVE_PACKS = "minimax_h3_fl2va_bf16 minimax_h3_turbo_lora" #@param {type:"string"}
MODEL_LOAD_MODE = "copy_to_local" #@param ["copy_to_local", "symlink"]
INSTALL_SAGEATTENTION = False #@param {type:"boolean"}
ENABLE_LTX_NODE = False #@param {type:"boolean"}

import subprocess
cmd = [
    'python', f'{REPO_DIR}/scripts/setup_env.py',
    '--repo-root', REPO_DIR,
    '--project', PROJECT_ID,
    '--mode', MODEL_LOAD_MODE,
    '--packs', *ACTIVE_PACKS.split(),
]
if INSTALL_SAGEATTENTION:
    cmd.append('--sageattention')
if ENABLE_LTX_NODE:
    cmd.append('--enable-ltx')

subprocess.run(cmd, check=True)


In [ ]:
#@title 3. 启动无头 ComfyUI 并按当前项目目录执行批处理队列
BATCH_NAME = "" #@param {type:"string"}
AUTO_DISCONNECT = True #@param {type:"boolean"}

import json, shlex, subprocess, time
from pathlib import Path

launch_info = json.loads(Path('/content/ComfyUI/launch_info.json').read_text(encoding='utf-8'))
comfy_dir = launch_info['comfyui_dir']
comfy_args = [a for a in shlex.split(launch_info['comfyui_args']) if a != '--normalvram']
if '--cache-none' not in comfy_args:
    comfy_args.append('--cache-none')
project_dir = Path(launch_info['project_dir'])
shared_dir = Path(launch_info['shared_dir'])

tasks_csv = project_dir / 'tasks.csv'
project_wf_dir = project_dir / 'workflows'
shared_wf_dir = shared_dir / 'workflows'
output_dir = project_dir / 'outputs'
log_file = project_dir / 'logs' / 'comfyui_headless.log'
log_file.parent.mkdir(parents=True, exist_ok=True)

print(f'执行项目: {launch_info["project_id"]} | 任务表: {tasks_csv}')

log_fp = log_file.open('a', encoding='utf-8')
server_proc = subprocess.Popen(
    ['python', 'main.py', *comfy_args],
    cwd=comfy_dir,
    stdout=log_fp,
    stderr=subprocess.STDOUT,
)

try:
    runner_cmd = [
        'python', f'{REPO_DIR}/scripts/batch_runner.py',
        '--tasks-csv', str(tasks_csv),
        '--workflows-dir', str(project_wf_dir),
        '--shared-workflows-dir', str(shared_wf_dir),
        '--output-dir', str(output_dir),
    ]
    if BATCH_NAME.strip():
        runner_cmd.extend(['--batch-name', BATCH_NAME.strip()])
    subprocess.run(runner_cmd, check=False)
finally:
    server_proc.terminate()
    log_fp.close()
    !sync
    time.sleep(15)
    if AUTO_DISCONNECT:
        print('[自动止损] 批处理完成并已同步至项目云端目录，即将自动释放 A100 运行时...')
        from google.colab import runtime
        runtime.unassign()
